# Stages 10–14 — Panel analysis figures

Run first: `python -m archive.impact_eda.impact_panel --through 14`

Outputs: `eda/outputs/impact_eda/figures/stage10_14/`

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "archive" / "impact_eda" / "impact_panel.py").exists():
    ROOT = ROOT.parents[2]
OUT = ROOT / "eda" / "outputs" / "impact_eda"
FIG = OUT / "figures" / "stage10_14"
FIG.mkdir(parents=True, exist_ok=True)

def save(fig, name):
    p = FIG / name
    fig.savefig(p, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("saved", p.relative_to(ROOT))

In [ ]:
# Within-county H2 coefficient
h2 = pd.read_csv(OUT / "stage12_within_county_ols_h2.csv")
fig, ax = plt.subplots(figsize=(5, 3))
row = h2[h2.term.str.contains("unusual")].iloc[0]
ax.barh(["log unusual px"], [row.coef], color="#2980b9")
ax.axvline(0, color="gray")
ax.set_title(f"H2 within FE (n={int(row.n)}): coef={row.coef:.3f}, p={row.pvalue:.3f}")
save(fig, "fig_h2_within_coef.png")

In [ ]:
# Regime break (unusual annual pixel-days)
ann = pd.read_csv(OUT / "stage11_annual_pixel_days.csv")
u = ann[ann.flood_type_label == "unusual"]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(u.year, u.flooded_pixel_count, marker="o")
ax.set_yscale("log")
ax.set_title("Unusual flood pixel-days by year (regime jump post-2020)")
ax.set_xlabel("Year")
save(fig, "fig_regime_jump.png")

In [ ]:
# H5 hydro drivers vs national unusual flood (annual Spearman)
hydro = pd.read_csv(OUT / "stage13_hydro_vs_flood_correlation.csv")
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(hydro.driver, hydro.spearman_r, color="#16a085")
ax.axvline(0, color="gray")
ax.set_title("H5: annual correlation with unusual flood pixel-days")
save(fig, "fig_h5_hydro_correlation.png")

In [ ]:
# H6 blind spots: high unusual flood, no OCHA assessed (top 15 county-years)
blind = pd.read_csv(OUT / "stage13_h6_blind_spots.csv").head(15)
fig, ax = plt.subplots(figsize=(8, 5))
labels = blind.adm2_pcode + " " + blind.year.astype(str)
ax.barh(labels, blind.unique_px, color="#c0392b")
ax.invert_yaxis()
ax.set_title("H6: high flood exposure, OCHA unassessed (top 15)")
save(fig, "fig_h6_blind_spots.png")

In [ ]:
# H8 / conflict: within-county flood vs type-2 events (month)
c = pd.read_csv(OUT / "stage14_flood_events_within_ols.csv")
row = c[c.term.str.contains("flood")].iloc[0]
fig, ax = plt.subplots(figsize=(5, 3))
ax.barh(["log flood px-days"], [row.coef], color="#8e44ad")
ax.axvline(0, color="gray")
ax.set_title(f"Co-occurrence (non-causal): coef={row.coef:.3f}, p={row.pvalue:.4f}, n={int(row.n)}")
save(fig, "fig_conflict_flood_cooccurrence.png")